In [1]:
import random
from itertools import combinations
from collections import Counter

# ---- Card representation ----
RANKS = list(range(2, 15))  # 2-14, 14 = Ace
SUITS = ['s', 'h', 'd', 'c']
DECK = [(r, s) for r in RANKS for s in SUITS]

# ---- Hand evaluation ----

def best_hand_rank(cards):
    best = None
    for combo in combinations(cards, min(5, len(cards))):
        rank = hand_rank(combo)
        if best is None or rank > best:
            best = rank
    return best

def hand_rank(cards):
    ranks = sorted([r for r, s in cards], reverse=True)
    suits = [s for r, s in cards]
    counts = Counter(ranks)
    rank_counts = sorted(counts.values(), reverse=True)

    is_flush = len(set(suits)) == 1
    is_straight = (len(set(ranks)) == 5 and ranks[0] - ranks[4] == 4)

    if set(ranks) == {14, 2, 3, 4, 5}:
        is_straight = True
        ranks = [5, 4, 3, 2, 1]
        counts = Counter(ranks)

    tiebreaker = sorted(ranks, key=lambda r: (counts[r], r), reverse=True)

    if is_straight and is_flush:
        return (8, tiebreaker)
    if rank_counts[0] == 4:
        return (7, tiebreaker)
    if rank_counts[:2] == [3, 2]:
        return (6, tiebreaker)
    if is_flush:
        return (5, tiebreaker)
    if is_straight:
        return (4, tiebreaker)
    if rank_counts[0] == 3:
        return (3, tiebreaker)
    if rank_counts[:2] == [2, 2]:
        return (2, tiebreaker)
    if rank_counts[0] == 2:
        return (1, tiebreaker)
    return (0, tiebreaker)

# ---- Single game ----

def play_game(n_picks=5, dealer_fill=8):
    """
    Player uses 'any' rule every turn — equivalent to just drawing
    random cards with no strategy. Dealer gets nothing from card
    passing (since 'any' always matches the first card), then fills
    to dealer_fill from the remaining deck.
    """
    deck = list(DECK)
    random.shuffle(deck)

    # Player takes first n_picks cards (any rule = first card always matches)
    my_hand = deck[:n_picks]
    remaining = deck[n_picks:]

    # Dealer fills to dealer_fill from remaining deck
    random.shuffle(remaining)
    dealer_hand = remaining[:dealer_fill]

    return 1 if best_hand_rank(my_hand) > best_hand_rank(dealer_hand) else 0

# ---- Simulate ----

def simulate(n_games=10000, n_picks=5, dealer_fill=8):
    wins = sum(play_game(n_picks, dealer_fill) for _ in range(n_games))
    return wins / n_games

if __name__ == '__main__':
    n_games = 10000
    print(f"Simulating random vs random over {n_games} games...")
    print(f"Player: 5 random cards (any rule every turn)")
    print(f"Dealer: fills to {8} cards from remaining deck, picks best 5\n")
    win_rate = simulate(n_games=n_games)
    print(f"Player win rate: {win_rate:.3f}")
    print(f"This is the true baseline — no rule mechanic, pure card draw advantage comparison")

Simulating random vs random over 10000 games...
Player: 5 random cards (any rule every turn)
Dealer: fills to 8 cards from remaining deck, picks best 5

Player win rate: 0.154
This is the true baseline — no rule mechanic, pure card draw advantage comparison
